# Τριμηνιαία Προσχηματική Κατάσταση Αποτελεσμάτων με το PROC COMPUTAB

## Σύνοψη για Στελέχη

Αυτό το σημειωματάριο κατασκευάζει την τριμηνιαία προσχηματική κατάσταση αποτελεσμάτων μιας περιφερειακής τράπεζας απευθείας από μηνιαία δεδομένα καθολικού χρησιμοποιώντας το PROC COMPUTAB, τη διαδικασία σύνταξης αναφορών σε πίνακες του SAS/ETS. Δρομολογεί τα έσοδα από τόκους, τα έξοδα τόκων, τα έσοδα από προμήθειες και τα λειτουργικά κόστη κάθε μήνα στη σωστή στήλη ημερολογιακού τριμήνου, και στη συνέχεια χρησιμοποιεί μπλοκ προγραμματισμού γραμμών για να υπολογίσει τα καθαρά έσοδα από τόκους, τα προ φόρων αποτελέσματα και τα καθαρά αποτελέσματα, και ένα μπλοκ στήλης για να αθροίσει τα τέσσερα τρίμηνα σε ένα σύνολο ολόκληρου του έτους.

## Πηγές Δεδομένων

Το μοναδικό συνθετικό σύνολο δεδομένων `bank_ledger` προσομοιώνει ένα οικονομικό έτος μηνιαίων γραμμών οικονομικών καταστάσεων για μια μεσαίου μεγέθους κοινοτική τράπεζα. Δώδεκα μηνιαίες παρατηρήσεις δημιουργούνται εσωτερικά με `call streaminit`/`rand` ώστε το σημειωματάριο να είναι πλήρως αυτόνομο.

| Μεταβλητή | Τύπος | Περιγραφή |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Ημερομηνία κατάστασης τέλους μήνα (Ιαν–Δεκ) |
| `loanint`  | Num | Τόκοι και προμήθειες από το χαρτοφυλάκιο δανείων (χιλιάδες USD) |
| `secint`   | Num | Τόκοι από το χαρτοφυλάκιο επενδυτικών τίτλων (χιλιάδες USD) |
| `depint`   | Num | Τόκοι που καταβάλλονται σε καταθέσεις (χιλιάδες USD) |
| `borrint`  | Num | Τόκοι που καταβάλλονται σε δανεισμό / προκαταβολές FHLB (χιλιάδες USD) |
| `feeinc`   | Num | Μη τοκοφόρα έσοδα (προμήθειες & χρεώσεις υπηρεσιών) (χιλιάδες USD) |
| `salaries` | Num | Μισθοί και έξοδα παροχών εργαζομένων (χιλιάδες USD) |
| `occupancy`| Num | Έξοδα εγκαταστάσεων και εξοπλισμού (χιλιάδες USD) |
| `othropex` | Num | Λοιπά μη τοκοφόρα λειτουργικά έξοδα (χιλιάδες USD) |
| `provision`| Num | Πρόβλεψη για πιστωτικές ζημίες (χιλιάδες USD) |
| `taxrate`  | Num | Πραγματικός φορολογικός συντελεστής επί των προ φόρων αποτελεσμάτων |

# Τριμηνιαία Προσχηματική Κατάσταση Αποτελεσμάτων με το PROC COMPUTAB

Οι ομάδες οικονομικών τραπεζών μετατρέπουν συνήθως ένα μηνιαίο γενικό καθολικό σε μια **τριμηνιαία κατάσταση αποτελεσμάτων** που δείχνει τα καθαρά έσοδα από τόκους και το τελικό καθαρό αποτέλεσμα. Το `PROC COMPUTAB` (SAS/ETS) είναι ειδικά σχεδιασμένο για αυτό: διατάσσει έναν προγραμματίσιμο πίνακα του οποίου οι **στήλες** είναι οι περίοδοι αναφοράς και οι **γραμμές** είναι οι γραμμές των καταστάσεων, και σας επιτρέπει να υπολογίζετε υποσύνολα με συνηθισμένη λογική βήματος DATA σε μπλοκ γραμμών και στηλών.

Σε αυτό το σημειωματάριο:

1. Δημιουργούμε ένα οικονομικό έτος συνθετικών μηνιαίων δεδομένων καθολικού για μια κοινοτική τράπεζα.
2. Δρομολογούμε κάθε μήνα στο ημερολογιακό του τρίμηνο και κατασκευάζουμε μια τριμηνιαία κατάσταση αποτελεσμάτων.
3. Υπολογίζουμε τα καθαρά έσοδα από τόκους, τα προ φόρων αποτελέσματα και τα καθαρά αποτελέσματα σε ένα **μπλοκ γραμμής**, και αθροίζουμε τα τρίμηνα σε ένα σύνολο ολόκληρου του έτους σε ένα **μπλοκ στήλης**.
4. Επαναχρησιμοποιούμε τον πίνακα `out=` ώστε η υπολογισμένη κατάσταση να μπορεί να τροφοδοτήσει μεταγενέστερη αναφορά.

## Βήμα 1 — Δημιουργία συνθετικών μηνιαίων δεδομένων καθολικού

Προσομοιώνουμε δώδεκα παρατηρήσεις τέλους μήνα. Τα έσοδα από τόκους δανείων και τίτλων ανεβαίνουν ήπια κατά τη διάρκεια του έτους, τα κόστη καταθέσεων και δανεισμού κλιμακώνονται με το περιβάλλον επιτοκίων, και τα έσοδα από προμήθειες, τα λειτουργικά έξοδα και η πρόβλεψη πιστωτικών ζημιών φέρουν ρεαλιστικό θόρυβο από μήνα σε μήνα. Το `call streaminit` σταθεροποιεί τον σπόρο ώστε τα δεδομένα να είναι αναπαραγώγιμα.

In [1]:
ΔΕΔΟΜΕΝΑ bank_ledger;
   CALL streaminit(20240115);
   ΜΟΡΦΗ stmtdate date9.;
   ΕΠΑΝΑΛΗΨΗ month = 1 ΕΩΣ 12;
      /* Ημερομηνία κατάστασης τέλους μήνα για το οικονομικό έτος 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Ήπια ανοδική τάση κατά τη διάρκεια του έτους + μηνιαίος θόρυβος */
      drift = 1 + 0.012 * (month - 1);

      /* Έσοδα από τόκους (χιλιάδες USD) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Έξοδα τόκων (χιλιάδες USD) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Μη τοκοφόρα έσοδα και έξοδα (χιλιάδες USD) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Πρόβλεψη για πιστωτικές ζημίες, περιστασιακά αυξημένη */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Πραγματικός φορολογικός συντελεστής */
      taxrate = 0.21;

      ΕΞΟΔΟΣ;
   ΤΕΛΟΣ;
   ΚΡΑΤΗΣΗ stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=bank_ledger noobs ΕΤΙΚΕΤΑ;
   TITLE 'Συνθετικό Μηνιαίο Καθολικό — Κοινοτική Τράπεζα, Οικ. Έτος 2024';
   ΕΤΙΚΕΤΑ stmtdate  = 'Ημ/νία Κατάστασης'
          loanint   = 'Τόκοι & Προμήθειες Δανείων'
          secint    = 'Τόκοι Τίτλων'
          depint    = 'Τόκοι Καταθέσεων'
          borrint   = 'Τόκοι Δανεισμού'
          feeinc    = 'Έσοδα Προμηθειών'
          salaries  = 'Μισθοί & Παροχές'
          occupancy = 'Εγκαταστάσεις & Εξοπλισμός'
          othropex  = 'Λοιπά Λειτουργικά Έξοδα'
          provision = 'Πρόβλεψη Ζημιών'
          taxrate   = 'Φορολογικός Συντελεστής';
   ΜΟΡΦΗ loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
ΕΚΤΕΛΕΣΗ;

                             Συνθετικό Μηνιαίο Καθολικό — Κοινοτική Τράπεζα, Οικ. Έτος 2024                             

               Ημ/νία Κατάστασης                        Τόκοι & Προμήθειες Δανείων             Τόκοι Τίτλων                 Τόκοι Καταθέσεων                Τόκοι Δανεισμού                 Έσοδα Προμηθειών               Μισθοί & Παροχές                         Εγκαταστάσεις & Εξοπλισμός                       Λοιπά Λειτουργικά Έξοδα                Πρόβλεψη Ζημιών                        Φορολογικός Συντελεστής
                       28JAN2024                                          1,772.95                   423.79                           531.47                         128.99                           339.33                         699.38                                             171.95                                        202.43                         130.93                                           0.21
                       28FEB2024                        


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## Βήμα 2 — Κατασκευή της τριμηνιαίας κατάστασης αποτελεσμάτων

Η καρδιά της αναφοράς είναι το βήμα `PROC COMPUTAB` παρακάτω.

- Το **`columns qtr1 qtr2 qtr3 qtr4 fy;`** ορίζει τέσσερις στήλες τριμήνων συν μια στήλη ολόκληρου του έτους.
- Το μη επισημασμένο **μπλοκ εισόδου** θέτει την αυτόματη μεταβλητή **`_col_`** με `qtr(stmtdate)`, που δρομολογεί κάθε μηνιαία παρατήρηση στη σωστή στήλη τριμήνου. Επειδή η είσοδος αναστρέφεται εξ ορισμού, κάθε μεταβλητή καθολικού προσγειώνεται στη γραμμή που μοιράζεται το όνομά της.
- Το **μπλοκ γραμμής** `inc_stmt:` εκτελείται μία φορά ανά στήλη και υπολογίζει τις παράγωγες γραμμές — καθαρά έσοδα από τόκους, συνολικά μη τοκοφόρα έξοδα, προ φόρων αποτελέσματα και καθαρά αποτελέσματα — ακριβώς όπως θα έκανε ένας λογιστής.
- Το **μπλοκ στήλης** `total:` εκτελείται μία φορά ανά γραμμή και αθροίζει τα τέσσερα τρίμηνα στη στήλη `fy` (ολόκληρου του έτους).

Οι εντολές `rows ... / ...` προσθέτουν τίτλους, εσοχές και γραμμές κανόνων (`ol` υπεργράμμιση, `ul` υπογράμμιση, `dul` διπλή υπογράμμιση) ώστε η έξοδος να διαβάζεται σαν πραγματική οικονομική κατάσταση.

In [2]:
TITLE 'Προσχηματική Κατάσταση Αποτελεσμάτων — Community Bancorp, Inc.';
title2 'Οικονομικό Έτος 2024  (Ποσά σε Χιλιάδες USD)';

ΔΙΑΔΙΚΑΣΙΑ computab ΔΕΔΟΜΕΝΑ=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Τέσσερα τρίμηνα συν μια συγκεντρωτική στήλη ολόκληρου του έτους */
   columns qtr1 qtr2 qtr3 qtr4 fy / ΜΟΡΦΗ=comma11.1;
   columns qtr1 / 'Τ1';
   columns qtr2 / 'Τ2';
   columns qtr3 / 'Τ3';
   columns qtr4 / 'Τ4';
   columns fy   / 'Πλήρες' 'Έτος' +3;

   /* Γραμμές κατάστασης αποτελεσμάτων, από πάνω προς τα κάτω */
   rows loanint  / 'Τόκοι & Προμήθειες Δανείων';
   rows secint   / 'Τόκοι Τίτλων'                   ul;
   rows intinc   / 'Σύνολο Εσόδων από Τόκους';
   rows depint   / 'Τόκοι Καταθέσεων';
   rows borrint  / 'Τόκοι Δανεισμού'                ul;
   rows intexp   / 'Σύνολο Εξόδων Τόκων';
   rows nii      / 'Καθαρά Έσοδα από Τόκους'        ol skip;
   rows provision/ 'Πρόβλεψη για Πιστωτικές Ζημίες' ul;
   rows niiap    / 'Καθαρά Έσοδα Τόκων μετά Πρόβλ.'  skip;
   rows feeinc   / 'Μη Τοκοφόρα Έσοδα'              ul;
   rows salaries / 'Μισθοί & Παροχές';
   rows occupancy/ 'Εγκαταστάσεις & Εξοπλισμός';
   rows othropex / 'Λοιπά Λειτουργικά Έξοδα'        ul;
   rows nonintexp/ 'Σύνολο Μη Τοκοφόρων Εξόδων'     skip;
   rows pretax   / 'Προ Φόρων Αποτελέσματα'         ol;
   rows taxexp   / 'Έξοδο Φόρου Εισοδήματος'        ul;
   rows netinc   / 'Καθαρά Αποτελέσματα'            dul;

   /* Μπλοκ εισόδου: δρομολόγηση κάθε μήνα στο ημερολογιακό του τρίμηνο */
   _col_ = qtr(stmtdate);

   /* Μπλοκ γραμμής: υπολογισμός υποσυνόλων κατάστασης σε κάθε στήλη */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Μπλοκ στήλης: άθροισμα των τεσσάρων τριμήνων στη στήλη ολόκληρου του έτους */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
ΕΚΤΕΛΕΣΗ;

                             Προσχηματική Κατάσταση Αποτελεσμάτων — Community Bancorp, Inc.                             
                                      Οικονομικό Έτος 2024  (Ποσά σε Χιλιάδες USD)                                      


                             The COMPUTAB Procedure                             

                             Τ1           Τ2           Τ3           Τ4       Πλήρες  
                                                                               Έτος  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Τόκοι & Προμήθειες Δανείων
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  Τόκοι Τίτλων
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  Σύνολο Εσόδων από Τόκους
 


NOTE: Option TITLE changed to Προσχηματική Κατάσταση Αποτελεσμάτων — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to Οικονομικό Έτος 2024  (Ποσά σε Χιλιάδες USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## Βήμα 3 — Επαναχρησιμοποίηση του συνόλου δεδομένων εξόδου του COMPUTAB

Το βήμα `PROC COMPUTAB` παραπάνω έγραψε τον υπολογισμένο πίνακά του στο `qtr_income` μέσω `out=`. Κάθε γραμμή αυτού του συνόλου δεδομένων είναι μια γραμμή κατάστασης και κάθε μεταβλητή στήλης (`qtr1`–`qtr4`, `fy`) κρατά την υπολογισμένη τιμή, οπότε μπορεί να τροφοδοτήσει μεταγενέστερη αναφορά. Παρακάτω τυπώνουμε τη συγκεντρωτική στήλη ολόκληρου του έτους για να επιβεβαιώσουμε ότι τα μεγέθη συμφωνούν.

In [3]:
TITLE 'Σύνολο Δεδομένων Εξόδου COMPUTAB — Στήλη Ολόκληρου του Έτους';

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=qtr_income ΕΤΙΚΕΤΑ noobs;
   ΜΕΤΑΒΛΗΤΗ _row_ fy;
   ΜΟΡΦΗ fy comma12.1;
   ΕΤΙΚΕΤΑ _row_ = 'Γραμμή' fy = 'Πλήρες Έτος';
ΕΚΤΕΛΕΣΗ;

TITLE;

                              Σύνολο Δεδομένων Εξόδου COMPUTAB — Στήλη Ολόκληρου του Έτους                              
                                      Οικονομικό Έτος 2024  (Ποσά σε Χιλιάδες USD)                                      


      Γραμμή            Πλήρες Έτος
------------  ---------------------
loanint                    23,588.4
secint                      5,416.8
intinc                     29,005.2
depint                      6,831.2
borrint                     1,650.7
intexp                      8,482.0
nii                        20,523.2
provision                   1,568.9
niiap                      18,954.3
feeinc                      3,703.2
salaries                    8,763.1
occupancy                   1,985.6
othropex                    2,944.8
nonintexp                  13,693.5
pretax                      8,964.1
taxexp                      1,882.5
netinc                      7,081.6




NOTE: Option TITLE changed to Σύνολο Δεδομένων Εξόδου COMPUTAB — Στήλη Ολόκληρου του Έτους.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Ερμηνεία των αποτελεσμάτων

Η προσχηματική κατάσταση διαβάζεται από πάνω προς τα κάτω σαν εποπτική αναφορά (call report) μιας τράπεζας: τα συνολικά έσοδα από τόκους μείον τα συνολικά έξοδα τόκων δίνουν τα **καθαρά έσοδα από τόκους** — εδώ περίπου \$20.5 εκατομμύρια για το έτος — τον κύριο μοχλό κερδοφορίας της τράπεζας. Αφαιρώντας την **πρόβλεψη για πιστωτικές ζημίες**, προσθέτοντας τα **έσοδα από προμήθειες** και συμψηφίζοντας τα **μη τοκοφόρα έξοδα** προκύπτουν προ φόρων αποτελέσματα περίπου \$9.0 εκατομμυρίων, και εφαρμόζοντας τον πραγματικό φορολογικό συντελεστή 21% προκύπτουν **καθαρά αποτελέσματα** κοντά στα \$7.1 εκατομμύρια. Το μπλοκ στήλης `total:` προσθέτει τα τέσσερα τρίμηνα στη στήλη ολόκληρου του έτους, οπότε τα ετήσια σύνολα συμφωνούν με το άθροισμα των τριμήνων εκ κατασκευής — επιβεβαιωμένο στο σύνολο δεδομένων `out=`, όπου το `netinc` ολόκληρου του έτους ύψους 7,081.6 ισούται με το άθροισμα των τεσσάρων τριμηνιαίων μεγεθών.

Επειδή τα πάντα υπολογίζονται μέσα στα προγραμματίσιμα μπλοκ γραμμών και στηλών του `PROC COMPUTAB`, η αντικατάσταση με ένα πραγματικό μηνιαίο καθολικό δεν απαιτεί καμία αλλαγή στη λογική της αναφοράς — αλλάζει μόνο το σύνολο δεδομένων εισόδου. Το σύνολο δεδομένων `out=` καθιστά στη συνέχεια την υπολογισμένη κατάσταση διαθέσιμη για πίνακες ελέγχου, ανάλυση τάσεων ή αυτοματοποίηση φακέλου διοικητικού συμβουλίου.